In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

ROOT_DIR = 'dataset'
NOISE_ROOT_DIR = 'dataset_noise'
digits = ['0', '1', '3', '8']
label_map = {'0': 0, '1': 1, '3': 2, '8': 3}

## Функции для загрузки тренировочных и тестовых данных 

In [ ]:
def load_train_data(samples_per_digit):
    X, y = [], []
    # берём по 400 вертикальных и горизонтальных изображений
    half_samples = samples_per_digit // 2
    for digit in digits:
        for direction in ['vertical', 'horizontal']:
            dir_name = os.path.join(ROOT_DIR, 'train', digit, direction)
            if not os.path.exists(dir_name):
                continue
            
            files = os.listdir(dir_name)[:half_samples]
            for filename in files:
                img_path = os.path.join(dir_name, filename)
                img = Image.open(img_path).convert('L') # делаем grayscale
                X.append(np.array(img))
                y.append(label_map[digit])
    
    X = np.array(X).astype('float32') / 255.0 # нормализуем
    X = np.expand_dims(X, axis=-1)
    y = keras.utils.to_categorical(np.array(y), num_classes=4)
    return X, y

def load_test_data(test_dir_name, base_dir=NOISE_ROOT_DIR):
    X, y = [], []
    for digit in digits:
        for direction in ['vertical', 'horizontal']:
            dir_name = os.path.join(base_dir, test_dir_name, digit, direction)
            if not os.path.exists(dir_name):
                continue
            for filename in os.listdir(dir_name):
                img_path = os.path.join(dir_name, filename)
                img = Image.open(img_path).convert('L')
                X.append(np.array(img))
                y.append(label_map[digit])
                
    X = np.array(X).astype('float32') / 255.0
    X = np.expand_dims(X, axis=-1)
    y = keras.utils.to_categorical(np.array(y), num_classes=4) # превращаем в массив вероятностей класса
    return X, y

## Загрузка датасетов

In [ ]:
test_sets_info = [
    ("test_1", 0, ROOT_DIR),
    ("test_2", 20, NOISE_ROOT_DIR),
    ("test_3", 50, NOISE_ROOT_DIR),
    ("test_4", 100, NOISE_ROOT_DIR),
    ("test_5", 200, NOISE_ROOT_DIR)
]

# загружаем тестовые выборки
test_datasets = {}
for name, noise, base in test_sets_info:
    X_t, y_t = load_test_data(name, base_dir=base)
    test_datasets[(name, noise)] = (X_t, y_t)

## Обучение моделей на датасетах

In [ ]:
train_sizes = [800, 600, 400, 200, 100]
results = []

# повторение обучения на различных размерах выборки
for size in train_sizes:
    print(f"\nРазмер выборки {size} картинок на цифру")
    
    # получаем тренировочную выборку
    X_data, y_data = load_train_data(size)
    
    # делим выборку на тренировочную и тестовую 15%
    X_train, X_val, y_train, y_val = train_test_split(
        X_data, y_data, test_size=0.15, stratify=y_data, random_state=42
    )
    
    # описываем архитектуру cnn
    model = keras.Sequential([
        layers.Input(shape=(100, 100, 1)), # размер входного изображения
        layers.Conv2D(32, (3, 3), activation='relu'), # окно поиска, ищет простые элементы
        layers.MaxPooling2D((2, 2)), # сжатие изображения
        layers.Conv2D(64, (3, 3), activation='relu'), # окно поиска, ищет элементы сложнее
        layers.MaxPooling2D((2, 2)), # сжатие изображения
        layers.Flatten(), # преобразование матрицы в вектор
        layers.Dense(64, activation='relu'), # выделение признаков
        layers.Dropout(0.3), # отсеивание 30 процентов нейронов
        layers.Dense(4, activation='softmax') # выходной слой, итоговая вероятность для каждого класса
    ])
    
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    
    model.fit(X_train, y_train, epochs=8, batch_size=32, validation_data=(X_val, y_val), verbose=0)
    
    for (test_name, noise_level), (X_t, y_t) in test_datasets.items():
        loss, acc = model.evaluate(X_t, y_t, verbose=0)
        
        results.append({
            "Трениров. выборка": size,
            "Количество пикселей с шумом": noise_level,
            "Точность": round(acc, 4)
        })
        print(f"[{test_name}] Шум {noise_level} px: Точность = {acc:.4f}")


Размер выборки 800 картинок на цифру


E0000 00:00:1777978207.535988   26074 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
W0000 00:00:1777978207.896747   26074 cpu_allocator_impl.cc:82] Allocation of 108800000 exceeds 10% of free system memory.
W0000 00:00:1777978209.597230   29806 cpu_allocator_impl.cc:82] Allocation of 39337984 exceeds 10% of free system memory.
W0000 00:00:1777978209.879155   29810 cpu_allocator_impl.cc:82] Allocation of 39337984 exceeds 10% of free system memory.
W0000 00:00:1777978210.053576   29806 cpu_allocator_impl.cc:82] Allocation of 39337984 exceeds 10% of free system memory.
W0000 00:00:1777978210.180504   29810 cpu_allocator_impl.cc:82] Allocation of 39337984 exceeds 10% of free system memory.


[test_1] Шум 0 px: Точность = 0.9750
[test_2] Шум 20 px: Точность = 0.9625
[test_3] Шум 50 px: Точность = 0.4250
[test_4] Шум 100 px: Точность = 0.2500
[test_5] Шум 200 px: Точность = 0.2500

Размер выборки 600 картинок на цифру
[test_1] Шум 0 px: Точность = 0.7250
[test_2] Шум 20 px: Точность = 0.6375
[test_3] Шум 50 px: Точность = 0.3625
[test_4] Шум 100 px: Точность = 0.2500
[test_5] Шум 200 px: Точность = 0.2500

Размер выборки 400 картинок на цифру
[test_1] Шум 0 px: Точность = 0.2375
[test_2] Шум 20 px: Точность = 0.2250
[test_3] Шум 50 px: Точность = 0.2875
[test_4] Шум 100 px: Точность = 0.1875
[test_5] Шум 200 px: Точность = 0.2375

Размер выборки 200 картинок на цифру
[test_1] Шум 0 px: Точность = 0.2000
[test_2] Шум 20 px: Точность = 0.1375
[test_3] Шум 50 px: Точность = 0.2000
[test_4] Шум 100 px: Точность = 0.1875
[test_5] Шум 200 px: Точность = 0.2250

Размер выборки 100 картинок на цифру
[test_1] Шум 0 px: Точность = 0.2375
[test_2] Шум 20 px: Точность = 0.3000
[test_3] 

## Полученные результаты

In [7]:
def show_result_table(results):
    df_results = pd.DataFrame(results)

    pivot_table = df_results.pivot(
        index="Трениров. выборка", 
        columns="Количество пикселей с шумом", 
        values="Точность"
    ).sort_index(ascending=False)

    print("Результаты, точность")
    display(pivot_table)

show_result_table(results)

Результаты, точность


Количество пикселей с шумом,0,20,50,100,200
Трениров. выборка,,,,,
800,0.9750,0.9625,0.4250,0.2500,0.2500
600,0.7250,0.6375,0.3625,0.2500,0.2500
400,0.2375,0.2250,0.2875,0.1875,0.2375
200,0.2000,0.1375,0.2000,0.1875,0.2250
100,0.2375,0.3000,0.2375,0.2500,0.2750


## Небольшая модель

In [8]:
results2 = []

# повторение обучения на различных размерах выборки
for size in train_sizes:
    print(f"\nРазмер выборки {size} картинок на цифру")
    
    # получаем тренировочную выборку
    X_data, y_data = load_train_data(size)
    
    # делим выборку на тренировочную и тестовую 15%
    X_train, X_val, y_train, y_val = train_test_split(
        X_data, y_data, test_size=0.15, stratify=y_data, random_state=42
    )
    
    # описываем архитектуру cnn
    model = keras.Sequential([
        layers.Input(shape=(100, 100, 1)),
        layers.Conv2D(16, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(32, activation='relu'),
        layers.Dense(4, activation='softmax')
    ])
    
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    
    model.fit(X_train, y_train, epochs=8, batch_size=32, validation_data=(X_val, y_val), verbose=0)
    
    for (test_name, noise_level), (X_t, y_t) in test_datasets.items():
        loss, acc = model.evaluate(X_t, y_t, verbose=0)
        
        results2.append({
            "Трениров. выборка": size,
            "Количество пикселей с шумом": noise_level,
            "Точность": round(acc, 4)
        })
        print(f"[{test_name}] Шум {noise_level} px: Точность = {acc:.4f}")


Размер выборки 800 картинок на цифру
[test_1] Шум 0 px: Точность = 0.2500
[test_2] Шум 20 px: Точность = 0.2500
[test_3] Шум 50 px: Точность = 0.2500
[test_4] Шум 100 px: Точность = 0.2500
[test_5] Шум 200 px: Точность = 0.2375

Размер выборки 600 картинок на цифру
[test_1] Шум 0 px: Точность = 0.2500
[test_2] Шум 20 px: Точность = 0.2500
[test_3] Шум 50 px: Точность = 0.2625
[test_4] Шум 100 px: Точность = 0.2125
[test_5] Шум 200 px: Точность = 0.1750

Размер выборки 400 картинок на цифру
[test_1] Шум 0 px: Точность = 0.2500
[test_2] Шум 20 px: Точность = 0.2500
[test_3] Шум 50 px: Точность = 0.2500
[test_4] Шум 100 px: Точность = 0.2500
[test_5] Шум 200 px: Точность = 0.2500

Размер выборки 200 картинок на цифру
[test_1] Шум 0 px: Точность = 0.2500
[test_2] Шум 20 px: Точность = 0.2500
[test_3] Шум 50 px: Точность = 0.2625
[test_4] Шум 100 px: Точность = 0.2375
[test_5] Шум 200 px: Точность = 0.2375

Размер выборки 100 картинок на цифру
[test_1] Шум 0 px: Точность = 0.2500
[test_2] 

In [9]:
show_result_table(results2)

Результаты, точность


Количество пикселей с шумом,0,20,50,100,200
Трениров. выборка,,,,,
800,0.25,0.25,0.2500,0.2500,0.2375
600,0.25,0.25,0.2625,0.2125,0.1750
400,0.25,0.25,0.2500,0.2500,0.2500
200,0.25,0.25,0.2625,0.2375,0.2375
100,0.25,0.25,0.2500,0.2500,0.2500
